# Multimodal Extraction — Images to Typed Python Objects

**Week 2 | Notebook 4 of 4**

**What you'll learn:**
- Outlines vision model setup
- Extracting tables from scanned invoices
- Structured output from charts and graphs
- Comparing extraction accuracy: Outlines vs. raw GPT-4V prompting
- Batch document processing pipeline

**Runtime:** ~40 minutes

**Note:** This notebook uses vision models. Ensure you have access to GPT-4o or GPT-4V.

In [ ]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("02_outlines/04_vision_structure.ipynb")

## 1. Setup

In [ ]:
import outlines
from outlines.inputs import Chat, Image
from PIL import Image as PILImage
from pydantic import BaseModel

from src.config import get_openai_client

client = get_openai_client()
model = outlines.from_openai(client, "gpt-4o")  # Vision model required

## 2. Extracting Tables from Scanned Invoices

In [ ]:
from PIL import ImageDraw

# For demo, we'll create a simple image programmatically
# In production, use: image = PILImage.open("invoice_scan.png")


class TableData(BaseModel):
    headers: list[str]
    rows: list[list[str]]


img = PILImage.new("RGB", (400, 200), color="white")
draw = ImageDraw.Draw(img)
draw.text((10, 10), "Invoice #1234", fill="black")
draw.text((10, 40), "Item    Qty    Price", fill="black")
draw.text((10, 70), "WidgetA   2     $50", fill="black")
draw.text((10, 100), "WidgetB   1     $30", fill="black")

img.save("./sample_invoice.png")

# Extract structured data from the image
prompt = Chat(
    [
        {"role": "system", "content": "Extract structured data from images."},
        {"role": "user", "content": ["Extract the table from this image:", Image(img)]},
    ]
)

result = model(prompt, TableData)
table = TableData.model_validate_json(result)

print(f"Headers: {table.headers}")
for row in table.rows:
    print(f"  {row}")

## 3. Structured Output from Charts and Graphs

In [ ]:
class ChartData(BaseModel):
    chart_type: str
    title: str
    x_axis_label: str
    y_axis_label: str
    data_points: list[dict]  # [{"label": "Q1", "value": 100}, ...]


# Create a simple bar chart image
chart_img = PILImage.new("RGB", (400, 300), color="white")
draw = ImageDraw.Draw(chart_img)
draw.text((10, 10), "Sales by Quarter", fill="black")
draw.text((50, 250), "Q1  Q2  Q3  Q4", fill="black")
draw.rectangle([(50, 150), (100, 250)], fill="blue")  # Q1: 100
draw.rectangle([(120, 100), (170, 250)], fill="blue")  # Q2: 150
draw.rectangle([(190, 80), (240, 250)], fill="blue")  # Q3: 170
draw.rectangle([(260, 120), (310, 250)], fill="blue")  # Q4: 130

chart_img.save("./sample_chart.png")

prompt = Chat([{"role": "user", "content": ["Describe this chart:", Image(chart_img)]}])

# For vision extraction, we might use the raw model first to understand the image
result = model(prompt, ChartData)
chart = ChartData.model_validate_json(result)

print(f"Chart type: {chart.chart_type}")
print(f"Title: {chart.title}")
print(f"Data points: {len(chart.data_points)}")

## 4. Comparing Extraction Accuracy: Outlines vs Raw GPT-4V

In [ ]:
# Raw prompting approach (no structure guarantee)
raw_response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Extract the table data as JSON with headers and rows"},
                {"type": "image_url", "image_url": {"url": "data:image/png;base64,..."}},
            ],
        }
    ],
)

print("Raw approach: May produce invalid JSON or markdown formatting")
print("Outlines approach: Guaranteed valid JSON matching schema")
print("\nFor vision tasks, Outlines works best with text-heavy images.")
print("For complex visual reasoning, raw prompting + manual validation may be needed.")

## 5. Batch Document Processing Pipeline

In [ ]:
class DocumentExtraction(BaseModel):
    document_type: str
    invoice_number: str
    total_amount: float
    date: str


# Process all images in a directory
# image_paths = glob("./documents/*.png")
image_paths = ["./sample_invoice.png"]  # Demo with one image

results = []
for path in image_paths:
    img = PILImage.open(path)
    prompt = Chat([{"role": "user", "content": ["Extract invoice data:", Image(img)]}])
    result = model(prompt, DocumentExtraction)
    doc = DocumentExtraction.model_validate_json(result)
    results.append(doc)
    print(f"Processed {path}: {doc.invoice_number}, ${doc.total_amount}")

print(f"\nTotal documents processed: {len(results)}")

## 6. Exercise: Extract Structured Data from Your Domain Images

1. Collect 5 images from your domain (receipts, forms, ID cards)
2. Define a Pydantic model for the structured data
3. Batch extract all images
4. Compare accuracy with manual transcription

In [ ]:
# YOUR TURN: Domain-specific image extraction

# class DomainDocument(BaseModel):
#     field1: str
#     field2: float
#     ...

# for path in glob("./my_documents/*.png"):
#     img = PILImage.open(path)
#     prompt = Chat([
#         {"role": "user", "content": ["Extract:", Image(img)]}
#     ])
#     result = model(prompt, DomainDocument)
#     doc = DomainDocument.model_validate_json(result)
#     print(doc)

---

**Week 2 Complete!** Next week: **Guidance** — Python-native control flow.